# CUDA-Only Unsloth Banking77 Generalization Training

This notebook fine-tunes an Unsloth LoRA model on [`tsilva/banking77`](https://huggingface.co/datasets/tsilva/banking77) for intent classification. The goal is no longer memorization; the goal is **test-set generalization**.

The data contract is:

- `system_prompt` -> system message
- `text` -> user prompt
- `label_text` -> assistant response

The training flow is:

- stratified train/validation split from the Hugging Face `train` split
- supervised fine-tuning with `SFTTrainer`
- early stopping on validation loss
- best-checkpoint restore
- deterministic generation on the Hugging Face `test` split
- exact-match test accuracy and W&B prediction logging

It is intentionally **CUDA-only**. If CUDA, Unsloth, TRL, W&B, or the model stack is missing, the notebook stops instead of falling back to CPU, MPS, or a toy local model.


## 1. Environment Contract

Run this in a Linux or Windows CUDA environment with Unsloth installed. The repo's Modal runner is the intended validation path for this notebook.

```bash
./run_modal.sh notebooks/unsloth-minimal-training.ipynb
```

W&B logging expects the Modal secret `wandb-secret` to provide `WANDB_API_KEY`.


### Configuration

The defaults favor accuracy over speed while staying within the Modal A10 runtime budget. `test_eval_limit=None` means the final accuracy is computed on the full Banking77 test split.


In [ ]:
CONFIG = {
    # Reproducibility
    'seed': 3407,

    # Data
    'dataset_name': 'tsilva/banking77',
    'train_split': 'train',
    'test_split': 'test',
    'val_fraction': 0.1,
    'train_limit': None,  # Set to an int for quick debugging; None uses the full train split
    'test_eval_limit': None,  # None evaluates all 3,076 test rows

    # Model
    'model_name': 'unsloth/gemma-3-1b-it-unsloth-bnb-4bit',
    'max_seq_length': 2048,
    'load_in_4bit': True,
    'load_in_8bit': False,
    'full_finetuning': False,

    # LoRA
    'lora_rank': 32,
    'lora_alpha': 32,
    'lora_dropout': 0,

    # Training
    'per_device_train_batch_size': 2,
    'gradient_accumulation_steps': 4,
    'max_steps': 1600,
    'learning_rate': 2e-4,
    'warmup_ratio': 0.03,
    'logging_steps': 25,
    'eval_steps': 200,
    'early_stopping_patience': 3,
    'throughput_token_sample_size': 512,  # Rows sampled to estimate training tokens/sec
    'val_generation_examples_per_label': 2,  # Stratified validation rows generated at eval time
    'val_generation_log_examples': 32,  # Example rows to keep in each W&B validation table
    'run_test_evaluation': False,  # Keep test set untouched until a finalist run
    'output_dir': 'outputs/unsloth-banking77-generalization',

    # Experiment tracking
    'wandb_project': 'unsloth-minimal-training',
    'wandb_run_name': 'unsloth-banking77-generalization',
    'target_gpu_name': 'Nvidia A10',
    'target_gpu_hourly_cost_usd': 1.1016,  # Modal A10: $0.000306/sec

    # Inference
    'max_new_tokens': 16,
}


### Hard CUDA and Package Checks

The shared helper prints package, Python, PyTorch, CUDA, cuDNN, and GPU details, then stops early if the runtime cannot support this notebook.


In [ ]:
import torch

from aiml_notebooks import require_environment

ENVIRONMENT = require_environment(
    packages=['unsloth', 'trl', 'datasets', 'transformers', 'bitsandbytes', 'wandb'],
    cuda=True,
    min_cuda_capability=(7, 0),
    error_prefix='CUDA fine-tuning environment check failed',
)


### Import Unsloth Primitives

Importing Unsloth before other model libraries lets it patch kernels and model classes as intended.


In [ ]:
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template, train_on_responses_only
from trl import SFTConfig, SFTTrainer


## 2. Load Banking77 and Build Splits

The Hugging Face `train` split is split again into train and validation partitions using a deterministic stratified split over the integer label column. The original Hugging Face `test` split is kept untouched for final accuracy.


In [ ]:
import random
from collections import defaultdict

from datasets import Dataset, load_dataset

raw_dataset = load_dataset(CONFIG['dataset_name'])
source_train = raw_dataset[CONFIG['train_split']]
test_dataset = raw_dataset[CONFIG['test_split']]

label_to_indices = defaultdict(list)
for index, label in enumerate(source_train['label']):
    label_to_indices[int(label)].append(index)

rng = random.Random(CONFIG['seed'])
train_indices = []
val_indices = []
for label, indices in sorted(label_to_indices.items()):
    shuffled = list(indices)
    rng.shuffle(shuffled)
    val_count = max(1, round(len(shuffled) * CONFIG['val_fraction']))
    val_indices.extend(shuffled[:val_count])
    train_indices.extend(shuffled[val_count:])

rng.shuffle(train_indices)
rng.shuffle(val_indices)

if CONFIG['train_limit'] is not None:
    train_indices = train_indices[:CONFIG['train_limit']]

train_dataset = source_train.select(train_indices)
val_dataset = source_train.select(val_indices)
if CONFIG['test_eval_limit'] is not None:
    test_dataset = test_dataset.select(range(CONFIG['test_eval_limit']))

LABELS = sorted(set(source_train['label_text']))
LABEL_SET = set(LABELS)

print(raw_dataset)
print(f'Train rows: {len(train_dataset):,}')
print(f'Validation rows: {len(val_dataset):,}')
print(f'Test rows: {len(test_dataset):,}')
print(f'Labels: {len(LABELS)}')
print(train_dataset[0])


def normalize_label(text: str) -> str:
    cleaned = text.strip().strip('`').strip().strip('.,;:!')
    first_token = cleaned.split()[0] if cleaned.split() else ''
    first_token = first_token.strip('`').strip().strip('.,;:!')
    if cleaned in LABEL_SET:
        return cleaned
    if first_token in LABEL_SET:
        return first_token
    return first_token


## 3. Load a Real Unsloth Model

The default model is Gemma 3 1B Instruct in Unsloth 4-bit form. That is still compact enough for the Modal A10, but more capable than the 270M smoke-test model.


In [ ]:
model, tokenizer = FastModel.from_pretrained(
    model_name=CONFIG['model_name'],
    max_seq_length=CONFIG['max_seq_length'],
    load_in_4bit=CONFIG['load_in_4bit'],
    load_in_8bit=CONFIG['load_in_8bit'],
    full_finetuning=CONFIG['full_finetuning'],
)

tokenizer = get_chat_template(
    tokenizer,
    chat_template='gemma3',
)


### Attach LoRA Adapters

This is the Unsloth adapter primitive. The base model stays frozen while selected projection modules receive trainable low-rank matrices.


In [ ]:
model = FastModel.get_peft_model(
    model,
    r=CONFIG['lora_rank'],
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    lora_alpha=CONFIG['lora_alpha'],
    lora_dropout=CONFIG['lora_dropout'],
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=CONFIG['seed'],
    use_rslora=False,
    loftq_config=None,
)


### Inspect Trainable Parameters

This confirms we are training a LoRA adapter instead of full fine-tuning the base model.


In [ ]:
trainable_params = 0
total_params = 0
for param in model.parameters():
    total_params += param.numel()
    if param.requires_grad:
        trainable_params += param.numel()

print(f'Trainable parameters: {trainable_params:,}')
print(f'Total parameters: {total_params:,}')
print(f'Trainable fraction: {100 * trainable_params / total_params:.3f}%')
assert 0 < trainable_params < total_params


## 4. Format the Dataset with the Chat Template

Every row becomes a three-message conversation: system instructions, user text, and assistant intent label.


In [ ]:
def to_messages(row):
    return {
        'messages': [
            {'role': 'system', 'content': row['system_prompt']},
            {'role': 'user', 'content': row['text']},
            {'role': 'assistant', 'content': row['label_text']},
        ]
    }

def apply_template(row):
    return {
        'text': tokenizer.apply_chat_template(
            row['messages'],
            tokenize=False,
            add_generation_prompt=False,
        ).removeprefix('<bos>')
    }

formatted_train_dataset = train_dataset.map(to_messages).map(apply_template)
formatted_val_dataset = val_dataset.map(to_messages).map(apply_template)
print(formatted_train_dataset[0]['text'][:1200])


## 5. Build the SFT Trainer with Early Stopping

Validation loss is evaluated every `eval_steps`. Early stopping watches that validation loss and `load_best_model_at_end=True` restores the best checkpoint before final test evaluation.


In [ ]:
import os
import time

import wandb
from transformers import EarlyStoppingCallback, TrainerCallback

os.environ['WANDB_PROJECT'] = CONFIG['wandb_project']
os.environ.setdefault('WANDB_LOG_MODEL', 'false')

throughput_sample_size = min(CONFIG['throughput_token_sample_size'], len(formatted_train_dataset))
throughput_texts = formatted_train_dataset.select(range(throughput_sample_size))['text']
throughput_token_lengths = [
    len(input_ids)
    for input_ids in tokenizer(
        throughput_texts,
        add_special_tokens=False,
        truncation=True,
        max_length=CONFIG['max_seq_length'],
    )['input_ids']
]
avg_train_tokens_per_sample = sum(throughput_token_lengths) / len(throughput_token_lengths)
wandb_run_name = f"{CONFIG['wandb_run_name']}-{time.strftime('%Y%m%d-%H%M%S', time.gmtime())}"
effective_batch_size = (
    CONFIG['per_device_train_batch_size'] * CONFIG['gradient_accumulation_steps']
)
effective_tokens_per_step = effective_batch_size * avg_train_tokens_per_sample
gpu_stats = torch.cuda.get_device_properties(0)
gpu_total_memory_gb = gpu_stats.total_memory / 1024**3


def select_stratified_rows(dataset, examples_per_label, seed):
    rows_by_label = defaultdict(list)
    for row in dataset:
        rows_by_label[row['label_text']].append(row)

    rng = random.Random(seed)
    selected_rows = []
    for label in sorted(rows_by_label):
        label_rows = rows_by_label[label]
        rng.shuffle(label_rows)
        selected_rows.extend(label_rows[:examples_per_label])
    rng.shuffle(selected_rows)
    return selected_rows


validation_generation_rows = select_stratified_rows(
    val_dataset,
    examples_per_label=CONFIG['val_generation_examples_per_label'],
    seed=CONFIG['seed'],
)


@torch.inference_mode()
def generate_label(model, tokenizer, row) -> str:
    messages = [
        {'role': 'system', 'content': row['system_prompt']},
        {'role': 'user', 'content': row['text']},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    ).removeprefix('<bos>')
    inputs = tokenizer(text, return_tensors='pt').to('cuda')
    prompt_length = inputs['input_ids'].shape[-1]
    outputs = model.generate(
        **inputs,
        max_new_tokens=CONFIG['max_new_tokens'],
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    generated_ids = outputs[0, prompt_length:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()


def macro_f1_score(expected_labels, predicted_labels):
    f1_scores = []
    for label in LABELS:
        true_positive = sum(
            expected == label and predicted == label
            for expected, predicted in zip(expected_labels, predicted_labels)
        )
        false_positive = sum(
            expected != label and predicted == label
            for expected, predicted in zip(expected_labels, predicted_labels)
        )
        false_negative = sum(
            expected == label and predicted != label
            for expected, predicted in zip(expected_labels, predicted_labels)
        )

        precision_denominator = true_positive + false_positive
        recall_denominator = true_positive + false_negative
        precision = true_positive / precision_denominator if precision_denominator else 0.0
        recall = true_positive / recall_denominator if recall_denominator else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        f1_scores.append(f1)
    return sum(f1_scores) / len(f1_scores)


wandb_run = wandb.init(
    project=CONFIG['wandb_project'],
    name=wandb_run_name,
    config=CONFIG,
)
wandb_run.config.update(
    {
        'wandb_actual_run_name': wandb_run_name,
        'throughput/avg_train_tokens_per_sample': avg_train_tokens_per_sample,
        'throughput/token_sample_size': throughput_sample_size,
        'throughput/effective_tokens_per_step': effective_tokens_per_step,
        'train/effective_batch_size': effective_batch_size,
        'hardware/gpu_name': gpu_stats.name,
        'hardware/gpu_total_memory_gb': gpu_total_memory_gb,
        'hardware/target_gpu_name': CONFIG['target_gpu_name'],
        'cost/gpu_hourly_cost_usd': CONFIG['target_gpu_hourly_cost_usd'],
        'val/generation_total': len(validation_generation_rows),
    },
    allow_val_change=True,
)
wandb_run_id = wandb_run.id
print(f"W&B run: https://wandb.ai/tsilva/{CONFIG['wandb_project']}/runs/{wandb_run_id}")
print(f'Estimated average train tokens/sample: {avg_train_tokens_per_sample:.1f}')
print(f'Effective tokens/optimizer step: {effective_tokens_per_step:.1f}')
print(f'Validation generation rows per eval: {len(validation_generation_rows)}')


def namespace_trainer_logs(logs):
    metric_name_map = {
        'loss': 'train/loss',
        'grad_norm': 'train/grad_norm',
        'learning_rate': 'train/lr',
        'epoch': 'train/epoch',
        'eval_loss': 'val/loss',
        'eval_runtime': 'val/runtime',
        'eval_samples_per_second': 'val/samples_per_second',
        'eval_steps_per_second': 'val/steps_per_second',
    }
    numeric_logs = {}
    for key, value in logs.items():
        if not isinstance(value, (int, float)):
            continue
        if key in metric_name_map:
            numeric_logs[metric_name_map[key]] = value
        elif '/' in key:
            numeric_logs[key] = value
        else:
            numeric_logs[f'trainer/{key}'] = value
    return numeric_logs


class WandbScalarLogger(TrainerCallback):
    def __init__(self, avg_tokens_per_sample, gpu_total_memory_gb, gpu_hourly_cost_usd):
        self.avg_tokens_per_sample = avg_tokens_per_sample
        self.last_train_log_step = None
        self.last_train_log_wall_time = None
        self.current_step_start_time = None
        self.accumulated_train_step_seconds = 0.0
        self.accumulated_train_steps = 0
        self.gpu_total_memory_gb = gpu_total_memory_gb
        self.gpu_hourly_cost_usd = gpu_hourly_cost_usd
        self.train_start_time = None

    def on_train_begin(self, args, state, control, **kwargs):
        self.last_train_log_step = state.global_step
        self.last_train_log_wall_time = time.perf_counter()
        self.train_start_time = self.last_train_log_wall_time

    def on_step_begin(self, args, state, control, **kwargs):
        self.current_step_start_time = time.perf_counter()

    def on_step_end(self, args, state, control, **kwargs):
        if self.current_step_start_time is None:
            return
        self.accumulated_train_step_seconds += time.perf_counter() - self.current_step_start_time
        self.accumulated_train_steps += 1
        self.current_step_start_time = None

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs or wandb.run is None:
            return
        numeric_logs = namespace_trainer_logs(logs)
        max_steps = int(getattr(state, 'max_steps', 0) or getattr(args, 'max_steps', 0) or 0)
        numeric_logs['train/global_step'] = state.global_step
        if max_steps > 0:
            numeric_logs['train/max_steps'] = max_steps
            numeric_logs['train/progress_percent'] = 100 * state.global_step / max_steps

        if 'loss' in logs:
            now = time.perf_counter()
            if self.last_train_log_wall_time is not None and self.last_train_log_step is not None:
                step_delta = state.global_step - self.last_train_log_step
                wall_elapsed_seconds = now - self.last_train_log_wall_time
                if (
                    self.accumulated_train_steps > 0
                    and self.accumulated_train_step_seconds > 0
                ):
                    world_size = max(1, int(getattr(args, 'world_size', 1) or 1))
                    effective_batch_size = (
                        args.per_device_train_batch_size
                        * args.gradient_accumulation_steps
                        * world_size
                    )
                    train_steps_per_second = (
                        self.accumulated_train_steps / self.accumulated_train_step_seconds
                    )
                    train_samples_per_second = (
                        self.accumulated_train_steps
                        * effective_batch_size
                        / self.accumulated_train_step_seconds
                    )
                    train_tokens_per_second = (
                        train_samples_per_second * self.avg_tokens_per_sample
                    )
                    price_per_second = self.gpu_hourly_cost_usd / 3600
                    numeric_logs.update(
                        {
                            'train/steps_per_second': train_steps_per_second,
                            'train/samples_per_second': train_samples_per_second,
                            'train/tokens_per_second': train_tokens_per_second,
                            'train/seconds_per_step': (
                                self.accumulated_train_step_seconds
                                / self.accumulated_train_steps
                            ),
                            'train/effective_batch_size': effective_batch_size,
                            'throughput/effective_tokens_per_step': (
                                effective_batch_size * self.avg_tokens_per_sample
                            ),
                        }
                    )
                    if price_per_second > 0:
                        numeric_logs['efficiency/train_tokens_per_dollar'] = (
                            train_tokens_per_second / price_per_second
                        )
                    self.accumulated_train_step_seconds = 0.0
                    self.accumulated_train_steps = 0
                if step_delta > 0 and wall_elapsed_seconds > 0:
                    numeric_logs['train/wall_steps_per_second'] = (
                        step_delta / wall_elapsed_seconds
                    )
            self.last_train_log_step = state.global_step
            self.last_train_log_wall_time = now

        if torch.cuda.is_available():
            reserved_gb = torch.cuda.memory_reserved() / 1024**3
            allocated_gb = torch.cuda.memory_allocated() / 1024**3
            peak_reserved_gb = torch.cuda.max_memory_reserved() / 1024**3
            peak_reserved_percent = 100 * peak_reserved_gb / self.gpu_total_memory_gb
            numeric_logs.update(
                {
                    'device_fit/vram_reserved_gb': reserved_gb,
                    'device_fit/vram_allocated_gb': allocated_gb,
                    'device_fit/peak_vram_reserved_gb': peak_reserved_gb,
                    'device_fit/peak_vram_reserved_percent': peak_reserved_percent,
                    'device_fit/vram_headroom_percent': 100 - peak_reserved_percent,
                }
            )

        if self.train_start_time is not None:
            elapsed_gpu_hours = (time.perf_counter() - self.train_start_time) / 3600
            numeric_logs['cost/elapsed_gpu_hours'] = elapsed_gpu_hours
            numeric_logs['cost/estimated_usd'] = elapsed_gpu_hours * self.gpu_hourly_cost_usd

        if numeric_logs:
            wandb.log(numeric_logs, step=state.global_step)


class ValidationGenerationLogger(TrainerCallback):
    def __init__(self, rows, tokenizer, max_logged_examples, gpu_hourly_cost_usd):
        self.rows = rows
        self.tokenizer = tokenizer
        self.max_logged_examples = max_logged_examples
        self.gpu_hourly_cost_usd = gpu_hourly_cost_usd
        self.train_start_time = None

    def on_train_begin(self, args, state, control, **kwargs):
        self.train_start_time = time.perf_counter()

    def on_evaluate(self, args, state, control, model=None, **kwargs):
        if model is None or wandb.run is None:
            return

        started_at = time.perf_counter()
        was_training = model.training
        model.eval()
        expected_labels = []
        predicted_labels = []
        example_rows = []

        for index, row in enumerate(self.rows):
            prediction = generate_label(model, self.tokenizer, row)
            normalized_prediction = normalize_label(prediction)
            expected = row['label_text']
            expected_labels.append(expected)
            predicted_labels.append(normalized_prediction)
            if len(example_rows) < self.max_logged_examples:
                example_rows.append(
                    [
                        state.global_step,
                        index,
                        row['text'],
                        expected,
                        prediction,
                        normalized_prediction,
                        normalized_prediction == expected,
                    ]
                )

        if was_training:
            model.train()

        total = len(expected_labels)
        correct = sum(
            expected == predicted
            for expected, predicted in zip(expected_labels, predicted_labels)
        )
        accuracy = correct / total if total else 0.0
        macro_f1 = macro_f1_score(expected_labels, predicted_labels) if total else 0.0
        out_of_label_predictions = sum(
            predicted not in LABEL_SET
            for predicted in predicted_labels
        )
        generation_runtime = time.perf_counter() - started_at

        logs = {
            'val/acc': accuracy,
            'val/macro_f1': macro_f1,
            'val/correct': correct,
            'val/total': total,
            'val/out_of_label_rate': out_of_label_predictions / total if total else 0.0,
            'val/generation_runtime': generation_runtime,
            'val/generation_samples_per_second': total / generation_runtime
            if generation_runtime > 0
            else 0.0,
        }
        if self.train_start_time is not None:
            estimated_cost = (
                (time.perf_counter() - self.train_start_time)
                / 3600
                * self.gpu_hourly_cost_usd
            )
            logs['cost/estimated_usd'] = estimated_cost
            if estimated_cost > 0:
                logs['efficiency/val_acc_per_usd'] = accuracy / estimated_cost
                logs['efficiency/val_macro_f1_per_usd'] = macro_f1 / estimated_cost

        wandb.log(logs, step=state.global_step)
        if example_rows:
            wandb.log(
                {
                    'val/generation_predictions': wandb.Table(
                        columns=[
                            'step',
                            'index',
                            'text',
                            'expected_response',
                            'prediction',
                            'normalized_prediction',
                            'exact_match',
                        ],
                        data=example_rows,
                    )
                },
                step=state.global_step,
            )


trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=formatted_train_dataset,
    eval_dataset=formatted_val_dataset,
    args=SFTConfig(
        dataset_text_field='text',
        per_device_train_batch_size=CONFIG['per_device_train_batch_size'],
        gradient_accumulation_steps=CONFIG['gradient_accumulation_steps'],
        max_steps=CONFIG['max_steps'],
        learning_rate=CONFIG['learning_rate'],
        warmup_ratio=CONFIG['warmup_ratio'],
        logging_steps=CONFIG['logging_steps'],
        eval_strategy='steps',
        eval_steps=CONFIG['eval_steps'],
        save_strategy='steps',
        save_steps=CONFIG['eval_steps'],
        save_total_limit=2,
        metric_for_best_model='eval_loss',
        greater_is_better=False,
        load_best_model_at_end=True,
        optim='adamw_8bit',
        weight_decay=0.001,
        lr_scheduler_type='linear',
        seed=CONFIG['seed'],
        output_dir=CONFIG['output_dir'],
        report_to='none',
        run_name=wandb_run_name,
    ),
    callbacks=[
        WandbScalarLogger(
            avg_train_tokens_per_sample,
            gpu_total_memory_gb,
            CONFIG['target_gpu_hourly_cost_usd'],
        ),
        ValidationGenerationLogger(
            validation_generation_rows,
            tokenizer,
            CONFIG['val_generation_log_examples'],
            CONFIG['target_gpu_hourly_cost_usd'],
        ),
        EarlyStoppingCallback(early_stopping_patience=CONFIG['early_stopping_patience']),
    ],
)


### Mask Prompt Tokens

This Unsloth helper trains only on assistant labels. The system and user messages still condition the model, but they do not contribute to the loss.


In [ ]:
trainer = train_on_responses_only(
    trainer,
    instruction_part='<start_of_turn>user\n',
    response_part='<start_of_turn>model\n',
)


### Show CUDA Memory Before Training

This gives us a concrete read on the GPU budget before the training loop starts.


In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory_gb = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
max_memory_gb = round(gpu_stats.total_memory / 1024**3, 3)
print(f'GPU: {gpu_stats.name}')
print(f'Max memory: {max_memory_gb} GB')
print(f'Reserved before training: {start_gpu_memory_gb} GB')


## 6. Train

The trainer evaluates validation loss periodically, early-stops when it stops improving, and restores the best checkpoint at the end.


In [ ]:
train_wall_start_time = time.perf_counter()
train_result = trainer.train()
train_wall_seconds = time.perf_counter() - train_wall_start_time

if wandb.run is not None:
    final_training_metrics = {
        f'train_result/{key}': value
        for key, value in train_result.metrics.items()
        if isinstance(value, (int, float))
    }
    if torch.cuda.is_available():
        final_peak_vram_gb = torch.cuda.max_memory_reserved() / 1024**3
        final_peak_vram_percent = 100 * final_peak_vram_gb / gpu_total_memory_gb
        final_training_metrics.update(
            {
                'train/wall_runtime_seconds': train_wall_seconds,
                'device_fit/final_peak_vram_reserved_gb': final_peak_vram_gb,
                'device_fit/final_peak_vram_reserved_percent': final_peak_vram_percent,
                'device_fit/final_vram_headroom_percent': 100 - final_peak_vram_percent,
                'cost/final_estimated_usd': (
                    train_wall_seconds / 3600 * CONFIG['target_gpu_hourly_cost_usd']
                ),
            }
        )
    wandb.log(final_training_metrics, step=train_result.global_step)

print(f'Best checkpoint: {trainer.state.best_model_checkpoint}')
print(f'Best validation loss: {trainer.state.best_metric}')
train_result


### Show CUDA Memory After Training

A small memory summary helps verify that the run stayed within the target GPU budget.


In [ ]:
end_gpu_memory_gb = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
used_for_training_gb = round(end_gpu_memory_gb - start_gpu_memory_gb, 3)
print(f'Reserved after training: {end_gpu_memory_gb} GB')
print(f'Additional reserved during training: {used_for_training_gb} GB')


## 7. Final Test Accuracy

The test set is intentionally optional. During experiment sweeps, validation generation metrics are the run-selection signal; reserve the test set for finalist runs.


In [ ]:
prediction_rows = []
correct = 0
test_accuracy = None

if CONFIG['run_test_evaluation']:
    for index, row in enumerate(test_dataset):
        prediction = generate_label(model, tokenizer, row)
        normalized_prediction = normalize_label(prediction)
        exact_match = normalized_prediction == row['label_text']
        correct += int(exact_match)
        prediction_rows.append(
            {
                'index': index,
                'text': row['text'],
                'expected_response': row['label_text'],
                'prediction': prediction,
                'normalized_prediction': normalized_prediction,
                'exact_match': exact_match,
            }
        )
        if (index + 1) % 250 == 0:
            print(f'Evaluated {index + 1:,}/{len(test_dataset):,} test rows')

    test_accuracy = correct / len(test_dataset)
    print(f'Test accuracy: {test_accuracy:.2%} ({correct:,}/{len(test_dataset):,})')

    for row in prediction_rows[:20]:
        print('-' * 80)
        print(f"Text:        {row['text']}")
        print(f"Expected:    {row['expected_response']}")
        print(f"Prediction:  {row['prediction']}")
        print(f"Normalized:  {row['normalized_prediction']}")
        print(f"Match:       {row['exact_match']}")
else:
    print("Skipped test evaluation. Set CONFIG['run_test_evaluation'] = True for finalist runs.")


### Log Test Predictions to W&B

The prediction table lets us inspect misses directly in W&B, while scalar metrics make runs comparable.


In [ ]:
if wandb.run is None:
    raise RuntimeError('W&B run is not active. Re-run the trainer setup cell before logging predictions.')

if CONFIG['run_test_evaluation']:
    predictions_table = wandb.Table(
        columns=[
            'index',
            'text',
            'expected_response',
            'prediction',
            'normalized_prediction',
            'exact_match',
        ]
    )
    for row in prediction_rows:
        predictions_table.add_data(
            row['index'],
            row['text'],
            row['expected_response'],
            row['prediction'],
            row['normalized_prediction'],
            row['exact_match'],
        )

    wandb.log({
        'test/predictions': predictions_table,
        'test/acc': test_accuracy,
        'test/correct': correct,
        'test/total': len(test_dataset),
        'best/val_loss': trainer.state.best_metric,
    })
    print(
        f"Logged {len(prediction_rows)} test predictions to W&B project "
        f"{CONFIG['wandb_project']} run {wandb_run_id}"
    )
else:
    wandb.log({
        'test/evaluation_skipped': 1,
        'best/val_loss': trainer.state.best_metric,
    })
    print('Logged that test evaluation was skipped.')


## 8. Save the LoRA Adapter

Saving only the adapter keeps the artifact small. Load it later on top of the same base model for inference or continued fine-tuning.


In [ ]:
adapter_dir = 'outputs/unsloth-banking77-generalization-lora'
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print(f'Saved LoRA adapter to {adapter_dir}')

import wandb

if wandb.run is not None:
    wandb.finish()


## Key Takeaways

- This notebook uses real Unsloth primitives and no fallback training path.
- The dataset source is `tsilva/banking77`: `system_prompt` -> system, `text` -> user, `label_text` -> assistant.
- The Hugging Face train split is split into train and validation partitions.
- Early stopping monitors validation loss and the best checkpoint is restored before test evaluation.
- Validation generation metrics are logged during training so experiment runs can be compared without tuning on the test split.
- Final test accuracy is optional and should be reserved for finalist configurations.
- W&B logs scalar training/eval metrics, throughput, device-fit, cost, and prediction tables.
